# Falcon Preimage Attack

---
**THIS IS NOT THE COMPLETE TUTORIAL - see file with `(MAIN)` in the name.**

---

First you'll need to select which hardware setup you have. You'll need to select a `SCOPETYPE`, a `PLATFORM`, and a `CRYPTO_TARGET`. `SCOPETYPE` can either be `'OPENADC'` for the CWLite/CW1200 or `'CWNANO'` for the CWNano. `PLATFORM` is the target device, with `'CWLITEARM'`/`'CW308_STM32F3'` being the best supported option, followed by `'CWLITEXMEGA'`/`'CW308_XMEGA'`, then by `'CWNANO'`. `CRYPTO_TARGET` selects the crypto implementation, with `'TINYAES128C'` working on all platforms. An alternative for `'CWLITEXMEGA'` targets is `'AVRCRYPTOLIB'`. For example:

```python
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
CRYPTO_TARGET='TINYAES128C'
SS_VER='SS_VER_1_1'
```

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
CRYPTO_TARGET='FALCON'
CRYPTO_OPTIONS='Falcon512C'
SS_VER='SS_VER_1_1'

The following code will build the firmware for the target.

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

(ChipWhisperer Other WARNING|File __init__.py:69) ChipWhisperer update available! See https://chipwhisperer.readthedocs.io/en/latest/installing.html for updating instructions
(ChipWhisperer NAEUSB WARNING|File naeusb.py:701) Your firmware (0.11) is outdated - latest is 0.62See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


INFO: Found ChipWhisperer😍


In [3]:
%%sh -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER" "$CRYPTO_OPTIONS"
cd ../../../hardware/victims/firmware/simpleserial-falcon
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 CRYPTO_OPTIONS=$4

Building for platform CW308_STM32F3 with CRYPTO_TARGET=FALCON
SS_VER set to SS_VER_1_1
C:/Users/DAVIDK~1/CHIPWH~1/cw/home/portable/avrgcc/bin/make clean_objs .dep 
make[1]: Entering directory 'C:/Users/davidking/ChipWhisperer5_64/cw/home/portable/chipwhisperer/hardware/victims/firmware/simpleserial-falcon'
Building for platform CW308_STM32F3 with CRYPTO_TARGET=FALCON
SS_VER set to SS_VER_1_1
rm -f -- simpleserial-falcon-CW308_STM32F3.hex
rm -f -- simpleserial-falcon-CW308_STM32F3.eep
rm -f -- simpleserial-falcon-CW308_STM32F3.cof
rm -f -- simpleserial-falcon-CW308_STM32F3.elf
rm -f -- simpleserial-falcon-CW308_STM32F3.map
rm -f -- simpleserial-falcon-CW308_STM32F3.sym
rm -f -- simpleserial-falcon-CW308_STM32F3.lss
rm -f -- objdir-CW308_STM32F3/*.o
rm -f -- objdir-CW308_STM32F3/*.lst
rm -f -- simpleserial-falcon.s fpr.s simpleserial.s stm32f3_hal.s stm32f3_hal_lowlevel.s stm32f3_sysmem.s
rm -f -- simpleserial-falcon.d fpr.d simpleserial.d stm32f3_hal.d stm32f3_hal_lowlevel.d stm32f3_sys

In [4]:
cw.program_target(scope, prog, "../../../hardware/victims/firmware/simpleserial-falcon/simpleserial-falcon-{}.hex".format(PLATFORM))

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 23003 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 23003 bytes


In [5]:
LOGN = 4 
DIM = (1 << LOGN)
nb_challenge = 100000
scope.adc.samples = 24000

In [6]:
import struct

def decompose_bytes(c):
    return bytearray(list(struct.pack('<d', c)))

def compose_float64(byte_list):
    return struct.unpack('<d', bytes(byte_list))[0] 

def get_trace(challenge):
    
    # setup coefficients of f(challenge)
    for idx in range(DIM):
        idx_hi = (idx >> 8) & 0xFF  # High byte
        idx_lo = idx & 0xFF         # Low byte


        text = bytearray([idx_hi, idx_lo]) + decompose_bytes(challenge[idx])
        scope.arm()

        target.simpleserial_write('k', text)
#         print([ord(chr(b)) for b in text])

        response = target.simpleserial_read('r', 8)
        response_ints = [ord(chr(b)) for b in response]
#         print("Response:", response_ints)
        
    # run scope
    test_idx = 15

    test_idx_hi = (test_idx >> 8) & 0xFF  # High byte
    test_idx_lo = test_idx & 0xFF         # Low byte
    target.simpleserial_write('p', bytearray([test_idx_hi, test_idx_lo]))

    response = target.simpleserial_read('r', 8)
#     response_ints = [ord(chr(b)) for b in response]
    response_ints = compose_float64(response)
#     print("check response:", response_ints)

    ret = scope.capture()
    if ret:
        print("Target timed out!")
        return []
    else:
        return scope.get_last_trace()

In [7]:
import h5py
from random import gauss, choices
import string

file = h5py.File("traces_cw.hdf5", "w")

key = [-1, 1, -1, -1, 1, -3, -11, 9, -2, 2, -8, -4, 2, 0, 1, -2]
logn = 4

key_group = file.create_group(''.join(choices(string.ascii_uppercase + string.digits, k=8)))
key_group.attrs.create("key", key)
key_group.attrs.create("logn", logn)

In [8]:
from tqdm.notebook import trange
import numpy as np

def get_random_challenge():
        from random import uniform
        return [uniform(-512, 512) for i in range(DIM)]

trace_array = []
challenge_array = []


for i in trange(nb_challenge, desc='Capturing traces'):
    challenge = get_random_challenge()
    trace = get_trace(challenge)

    if trace is not None and len(trace) > 0:
        dataset = key_group.create_dataset(name=str(i), data=trace, dtype=np.float64)
        dataset.attrs.create("challenge", challenge)

file.close()

Capturing traces:   0%|          | 0/500 [00:00<?, ?it/s]

We don't need the hardware anymore, so we'll disconnect:

In [28]:
scope.dis()
target.dis()